# Residual LSTM experiments

This notebook reads the completed 15-run experiment; it does not train models. See `docs/MILESTONE_5.md` for methodology, limitations and reproduction. The 2025 test period remains reserved.

In [ ]:
from pathlib import Path
import json
import pandas as pd
from IPython.display import display, Image

ROOT = Path.cwd() if (Path.cwd() / "docs").exists() else Path.cwd().parent
results = json.loads((ROOT / "docs/lstm_results_v1.json").read_text())
assert results["attempts"] == 15 and not results["test_evaluated"]

## Prediction and temporal design

For each 30-session input, the target is next Close minus origin Close. The network predicts the standardised correction; its inverse transform plus origin Close yields the USD price forecast. Input and correction statistics use fitting observations only. Fit through 2021 and validate internally in 2022 for selection; then fit fresh models through 2022 for 19 fixed epochs and compare on 2023–2024. No external early stopping is used.

In [ ]:
display(pd.DataFrame(results["selection"]["groups"]))
display(pd.DataFrame(results["internal_runs"])[["configuration_id", "seed", "best_epoch", "epochs_completed", "evaluation_metrics"]])

D has the lowest mean internal MAE, but all configurations are within the predefined 1% band. After preferring fewer units, C wins by lower seed standard deviation. This rule selects 16 units, dropout 0.2, learning rate 0.001 and batch size 32. Median best epoch 19 fixes the subsequent refits; no best-seed selection is performed.

In [ ]:
display(Image(filename=str(ROOT / "artifacts/lstm_v1/figures/selection_curves.png")))
display(Image(filename=str(ROOT / "artifacts/lstm_v1/figures/validation_curves.png")))

The full plot uses logarithmic loss axes; detail panels have individual linear scales. Training inference and validation MSE use the same epoch weights with dropout disabled. Ordinary minibatch loss uses changing weights and active dropout for C. A large loss gap partly reflects larger internal-period dollar movements; it is not by itself proof of overfitting. Post-minimum validation deterioration alongside falling training loss suggests mild overfitting or optimisation drift in some runs. No clear underfitting diagnosis follows.

In [ ]:
display(Image(filename=str(ROOT / "artifacts/lstm_v1/figures/final_training_curves.png")))
rows = [{"model": name, **metrics} for name, metrics in results["baselines"].items()]
rows += [{"model": f"LSTM C seed {run["seed"]}", **run["evaluation_metrics"]} for run in results["final_runs"]]
display(pd.DataFrame(rows))
display(pd.DataFrame(results["external_aggregate"]))
display(Image(filename=str(ROOT / "artifacts/lstm_v1/figures/benchmark_comparison.png")))

Every LSTM seed is worse than naive on MAE and RMSE across the same 502 dates. Mean MAE is approximately 0.92% worse. These results do not justify expanding the search or selecting a favourable seed. Error bars are sample standard deviations over three runs, not confidence intervals. Feature experiments remain a separate, unimplemented milestone.

In [ ]:
display(pd.DataFrame([{ "seed": run["seed"], "year": year, **metrics}
                      for run in results["final_runs"] for year, metrics in run["by_year"].items()]))
display(results["internal_scaling"])
display(results["final_scaling"])